[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/advanced-ab-testing/blob/main/study_notes/week03_sample_size_mde.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 3: Sample Size Estimation & Minimum Detectable Effect

**Course**: Advanced A/B Testing | **Context**: QuickCart (online electronics retailer)

**Your role**: Data scientist on the experimentation platform team. Before launching any experiment, you need to answer: "How many users do we need?" and "What's the smallest improvement we can reliably detect?" Getting this wrong means either wasting traffic on underpowered tests or running experiments far longer than necessary.

---

## What You'll Learn

1. Why pre-experiment sample size calculation is essential
2. Deriving the sample size formula from first principles (power analysis)
3. Implementing a reusable `estimate_sample_size()` function
4. How sample size depends on alpha, power, effect size, and variance
5. Minimum Detectable Effect (MDE) — inverting the sample size formula
6. Practical planning: what can QuickCart detect with its traffic?
7. Variance reduction as a lever to shrink required sample size
8. Outlier filtering (winsorization) as a crude but effective variance reduction technique

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---

## 1. The Intuition Behind Sample Size

### The Telescope Analogy

Think of an A/B test as a **telescope**. The effect size (how much your treatment improves the metric) is the star you're trying to see. The sample size determines the **resolution** of your telescope.

- **Large effect** (bright star): A small telescope (small sample) can see it clearly.
- **Small effect** (dim star): You need a powerful telescope (large sample) to distinguish it from background noise.
- **High variance** (atmospheric turbulence): Makes everything blurry — you need even more resolution.

### Why Pre-Calculate?

Without a sample size calculation, you face two risks:

1. **Underpowered test** (too few users): You run for 2 weeks, see "no significant difference," and kill a feature that actually works. You've wasted engineering time AND lost revenue.

2. **Overpowered test** (too many users): You expose 500K users to a potentially bad experience when 50K would have given you an answer. You've wasted traffic and slowed down your experimentation velocity.

### The QuickCart Scenario

QuickCart's product team wants to test a new recommendation algorithm. They believe it could increase average order value by 2-5%. The experimentation team needs to answer:

- How many users per group to detect a 3% lift?
- With 50,000 daily users, how long will the experiment take?
- What's the smallest effect they can detect in a 2-week window?

---

## 2. Deriving the Sample Size Formula

### Starting Point: The Hypothesis Test

We are testing:
- $H_0$: $\mu_T - \mu_C = 0$ (no effect)
- $H_1$: $\mu_T - \mu_C = \delta$ (true effect is $\delta$)

Under the null, the test statistic for a two-sample z-test (large samples) is:

$$Z = \frac{\bar{X}_T - \bar{X}_C}{\sqrt{\frac{\sigma^2}{n_T} + \frac{\sigma^2}{n_C}}}$$

For equal group sizes ($n_T = n_C = n$), this simplifies to:

$$Z = \frac{\bar{X}_T - \bar{X}_C}{\sigma\sqrt{2/n}}$$

### The Power Condition

We want two things simultaneously:
1. **Type I error control**: Reject $H_0$ only when $|Z| > z_{\alpha/2}$ (for two-sided test)
2. **Power**: When the true effect is $\delta$, we detect it with probability $1 - \beta$

### The Key Formula

Combining these requirements yields the **sample size per group**:

$$\boxed{n = \frac{(z_{\alpha/2} + z_{\beta})^2 \cdot 2\sigma^2}{\delta^2}}$$

where:
- $z_{\alpha/2}$ = critical value for significance level (1.96 for $\alpha = 0.05$)
- $z_{\beta}$ = critical value for power (0.84 for power = 80%)
- $\sigma^2$ = variance of the metric
- $\delta$ = minimum effect size to detect (absolute difference)

:::{admonition} Full Derivation of the Sample Size Formula
:class: dropdown

**Step 1: Distribution under the null**

Under $H_0$ ($\delta = 0$), the test statistic $Z \sim N(0, 1)$.

We reject when $|Z| > z_{\alpha/2}$, i.e., when:
$$\frac{|\bar{X}_T - \bar{X}_C|}{\sigma\sqrt{2/n}} > z_{\alpha/2}$$

**Step 2: Distribution under the alternative**

Under $H_1$ ($\mu_T - \mu_C = \delta$), the difference $\bar{X}_T - \bar{X}_C \sim N(\delta, 2\sigma^2/n)$.

Therefore, the test statistic has a shifted distribution:
$$Z \sim N\left(\frac{\delta}{\sigma\sqrt{2/n}}, 1\right)$$

**Step 3: Power requirement**

Power = $P(\text{reject } H_0 | H_1 \text{ is true}) = 1 - \beta$

For a one-sided argument (considering the right tail):
$$P\left(Z > z_{\alpha/2} \mid Z \sim N\left(\frac{\delta}{\sigma\sqrt{2/n}}, 1\right)\right) = 1 - \beta$$

This means:
$$\frac{\delta}{\sigma\sqrt{2/n}} - z_{\alpha/2} = z_{\beta}$$

Wait — here $z_{\beta}$ is the z-value such that $P(Z > -z_{\beta}) = 1 - \beta$, which means $z_{\beta} = z_{1-\beta}$ (the $(1-\beta)$ quantile of the standard normal). For 80% power, $z_{\beta} = 0.842$.

**Step 4: Solve for n**

$$\frac{\delta}{\sigma\sqrt{2/n}} = z_{\alpha/2} + z_{\beta}$$

$$\frac{\delta \sqrt{n}}{\sigma\sqrt{2}} = z_{\alpha/2} + z_{\beta}$$

$$\sqrt{n} = \frac{(z_{\alpha/2} + z_{\beta}) \cdot \sigma\sqrt{2}}{\delta}$$

$$n = \frac{(z_{\alpha/2} + z_{\beta})^2 \cdot 2\sigma^2}{\delta^2}$$

This is the required sample size **per group** for a two-sample test with equal allocation.

**Numerical example**: For $\alpha = 0.05$, power = 80%, $\sigma = 1$, $\delta = 0.2$:

$$n = \frac{(1.96 + 0.84)^2 \cdot 2 \cdot 1}{0.04} = \frac{7.84 \cdot 2}{0.04} = 392 \text{ per group}$$
:::

In [ ]:
# Quick verification of the formula with known values
alpha = 0.05
beta = 0.20  # power = 1 - beta = 0.80
sigma = 1.0
delta = 0.2  # effect size (absolute)

z_alpha_half = stats.norm.ppf(1 - alpha / 2)  # 1.96
z_beta = stats.norm.ppf(1 - beta)             # 0.84

n_per_group = (z_alpha_half + z_beta)**2 * 2 * sigma**2 / delta**2

print(f"z_alpha/2 = {z_alpha_half:.4f}")
print(f"z_beta    = {z_beta:.4f}")
print(f"")
print(f"Required n per group: {n_per_group:.0f}")
print(f"Total sample size:    {2 * n_per_group:.0f}")
print(f"")
print(f"Interpretation: To detect an effect of {delta} (with sigma={sigma}),")
print(f"you need ~{n_per_group:.0f} observations per group at 80% power, alpha=0.05.")

---

## 3. Implementing Sample Size Estimation

Now let's build a practical function that takes **real data** and computes the required sample size for a list of hypothetical effect sizes. This is what you'd actually use in production.

The function:
1. Estimates $\sigma^2$ from the data
2. Converts relative effect sizes to absolute differences
3. Computes required $n$ per group for each effect size

In [ ]:
def estimate_sample_size(df, metric_name, effects, alpha=0.05, beta=0.2):
    """
    Estimate required sample size per group for detecting given effect sizes.
    
    Parameters
    ----------
    df : pd.DataFrame
        Historical data containing the metric.
    metric_name : str
        Column name for the metric of interest.
    effects : list of float
        Relative effect sizes to test (e.g., [0.01, 0.02, 0.05] for 1%, 2%, 5% lifts).
    alpha : float
        Significance level (Type I error rate). Default 0.05.
    beta : float
        Type II error rate (1 - power). Default 0.2 (80% power).
    
    Returns
    -------
    pd.DataFrame
        Table with effect sizes and corresponding required sample sizes.
    """
    # Estimate parameters from historical data
    metric_values = df[metric_name].values
    mu = np.mean(metric_values)
    sigma_sq = np.var(metric_values, ddof=1)
    
    # Critical values
    z_alpha_half = stats.norm.ppf(1 - alpha / 2)
    z_beta = stats.norm.ppf(1 - beta)
    
    results = []
    for effect in effects:
        # Convert relative effect to absolute difference
        delta = mu * effect
        
        # Sample size formula
        n_per_group = (z_alpha_half + z_beta)**2 * 2 * sigma_sq / delta**2
        n_per_group = int(np.ceil(n_per_group))
        
        results.append({
            'effect_size_pct': f"{effect*100:.1f}%",
            'absolute_delta': round(delta, 4),
            'n_per_group': n_per_group,
            'total_n': 2 * n_per_group
        })
    
    result_df = pd.DataFrame(results)
    
    # Print summary
    print(f"Metric: {metric_name}")
    print(f"  Mean (mu):     {mu:.4f}")
    print(f"  Std (sigma):   {np.sqrt(sigma_sq):.4f}")
    print(f"  Variance:      {sigma_sq:.4f}")
    print(f"  Alpha:         {alpha}")
    print(f"  Power:         {1 - beta:.0%}")
    print(f"  CV (sigma/mu): {np.sqrt(sigma_sq)/mu:.4f}")
    print()
    
    return result_df

In [ ]:
# Generate synthetic QuickCart order data
# Revenue per order: log-normal distribution (realistic for e-commerce)
np.random.seed(42)
n_historical = 10000

# Log-normal parameters chosen so that mean ~ $75, with right skew
log_mu = 4.0
log_sigma = 0.7
revenue = np.random.lognormal(mean=log_mu, sigma=log_sigma, size=n_historical)

df_historical = pd.DataFrame({'order_revenue': revenue})

print("QuickCart Historical Revenue Data")
print("=" * 40)
print(df_historical['order_revenue'].describe())
print(f"\nSkewness: {stats.skew(revenue):.2f}")
print(f"Kurtosis: {stats.kurtosis(revenue):.2f}")

In [ ]:
# Visualize the distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(revenue, bins=80, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(np.mean(revenue), color='red', linestyle='--', linewidth=2, label=f'Mean = ${np.mean(revenue):.2f}')
axes[0].axvline(np.median(revenue), color='green', linestyle='--', linewidth=2, label=f'Median = ${np.median(revenue):.2f}')
axes[0].set_xlabel('Order Revenue ($)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Order Revenue')
axes[0].legend()
axes[0].set_xlim(0, 400)

axes[1].hist(revenue, bins=80, edgecolor='black', alpha=0.7, color='steelblue', cumulative=True, density=True)
axes[1].set_xlabel('Order Revenue ($)')
axes[1].set_ylabel('Cumulative Proportion')
axes[1].set_title('CDF of Order Revenue')
axes[1].axhline(0.95, color='red', linestyle=':', alpha=0.7, label='95th percentile')
axes[1].axhline(0.99, color='orange', linestyle=':', alpha=0.7, label='99th percentile')
axes[1].legend()
axes[1].set_xlim(0, 400)

plt.tight_layout()
plt.show()

In [ ]:
# Compute sample sizes for various effect sizes
effects = [0.01, 0.02, 0.03, 0.05, 0.10]
result = estimate_sample_size(df_historical, 'order_revenue', effects)
result

**Interpretation**: To detect a 2% increase in average order revenue ($75 -> $76.50, a $1.50 lift), QuickCart needs approximately ~70K-100K+ orders per group depending on the variance. The high coefficient of variation (CV) of revenue data makes it expensive to detect small relative effects.

:::{admonition} The Coefficient of Variation (CV) Problem
:class: dropdown

The required sample size scales as:

$$n \propto \frac{\sigma^2}{\delta^2} = \frac{\sigma^2}{(\mu \cdot \text{effect})^2} = \frac{1}{\text{effect}^2} \cdot \left(\frac{\sigma}{\mu}\right)^2 = \frac{\text{CV}^2}{\text{effect}^2}$$

So the sample size depends on the **ratio** of the coefficient of variation to the effect size. For revenue data with CV ~ 0.8, detecting a 2% effect requires:

$$n \propto \frac{0.8^2}{0.02^2} = \frac{0.64}{0.0004} = 1600$$

times the base factor $(z_{\alpha/2} + z_\beta)^2 \cdot 2 \approx 15.7$.

This is why **variance reduction** is so valuable: reducing CV from 0.8 to 0.6 cuts the required sample size by 44%!
:::

---

## 4. Interactive Exploration

Let's visualize how the required sample size changes as we vary each parameter. This builds intuition for what drives experiment duration.

In [ ]:
def compute_n(alpha, beta, sigma, delta):
    """Compute sample size per group given parameters."""
    z_a = stats.norm.ppf(1 - alpha / 2)
    z_b = stats.norm.ppf(1 - beta)
    return (z_a + z_b)**2 * 2 * sigma**2 / delta**2


fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) Effect size
effects_range = np.linspace(0.01, 0.15, 100)
mu_base = 75.0
sigma_base = 55.0
n_vs_effect = [compute_n(0.05, 0.2, sigma_base, mu_base * e) for e in effects_range]

axes[0, 0].plot(effects_range * 100, n_vs_effect, 'b-', linewidth=2)
axes[0, 0].axhline(50000, color='red', linestyle='--', alpha=0.7, label='50K users (1 day @ QuickCart)')
axes[0, 0].axhline(350000, color='orange', linestyle='--', alpha=0.7, label='350K users (1 week)')
axes[0, 0].set_xlabel('Relative Effect Size (%)')
axes[0, 0].set_ylabel('Required n per group')
axes[0, 0].set_title('(a) Sample Size vs Effect Size')
axes[0, 0].set_ylim(0, 500000)
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# (b) Alpha (significance level)
alphas = np.linspace(0.001, 0.20, 100)
n_vs_alpha = [compute_n(a, 0.2, sigma_base, mu_base * 0.03) for a in alphas]

axes[0, 1].plot(alphas, n_vs_alpha, 'g-', linewidth=2)
axes[0, 1].axvline(0.05, color='red', linestyle='--', alpha=0.7, label='alpha = 0.05 (standard)')
axes[0, 1].axvline(0.01, color='orange', linestyle='--', alpha=0.7, label='alpha = 0.01 (strict)')
axes[0, 1].set_xlabel('Alpha (significance level)')
axes[0, 1].set_ylabel('Required n per group')
axes[0, 1].set_title('(b) Sample Size vs Alpha (effect=3%)')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# (c) Power (1 - beta)
powers = np.linspace(0.50, 0.99, 100)
n_vs_power = [compute_n(0.05, 1 - p, sigma_base, mu_base * 0.03) for p in powers]

axes[1, 0].plot(powers * 100, n_vs_power, 'r-', linewidth=2)
axes[1, 0].axvline(80, color='blue', linestyle='--', alpha=0.7, label='80% power (standard)')
axes[1, 0].axvline(90, color='purple', linestyle='--', alpha=0.7, label='90% power')
axes[1, 0].set_xlabel('Power (%)')
axes[1, 0].set_ylabel('Required n per group')
axes[1, 0].set_title('(c) Sample Size vs Power (effect=3%)')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# (d) Variance (sigma)
sigmas = np.linspace(20, 100, 100)
n_vs_sigma = [compute_n(0.05, 0.2, s, mu_base * 0.03) for s in sigmas]

axes[1, 1].plot(sigmas, n_vs_sigma, 'm-', linewidth=2)
axes[1, 1].axvline(sigma_base, color='red', linestyle='--', alpha=0.7, label=f'Current sigma={sigma_base}')
axes[1, 1].axvline(sigma_base / np.sqrt(2), color='green', linestyle='--', alpha=0.7, 
                    label=f'Halved variance (sigma={sigma_base/np.sqrt(2):.0f})')
axes[1, 1].set_xlabel('Standard Deviation (sigma)')
axes[1, 1].set_ylabel('Required n per group')
axes[1, 1].set_title('(d) Sample Size vs Variance (effect=3%)')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Key Observations

1. **Effect size** has the strongest influence: $n \propto 1/\delta^2$. Halving the effect you want to detect **quadruples** the required sample.

2. **Alpha** has moderate influence: Going from 0.05 to 0.01 increases required n by ~50%.

3. **Power** matters most at the high end: Going from 80% to 90% power adds ~30% more sample. Going from 90% to 99% nearly doubles it.

4. **Variance** is linear: $n \propto \sigma^2$. This is the lever you can actually pull through variance reduction techniques (Weeks 4-5).

---

## 5. Minimum Detectable Effect (MDE)

### Inverting the Question

Often in practice, the sample size is **fixed** (by traffic and time constraints). The question becomes:

> "Given $n$ users available, what's the smallest effect I can reliably detect?"

This is the **Minimum Detectable Effect (MDE)**.

### Deriving the MDE Formula

Starting from the sample size formula and solving for $\delta$:

$$n = \frac{(z_{\alpha/2} + z_{\beta})^2 \cdot 2\sigma^2}{\delta^2}$$

$$\delta^2 = \frac{(z_{\alpha/2} + z_{\beta})^2 \cdot 2\sigma^2}{n}$$

$$\boxed{\text{MDE} = (z_{\alpha/2} + z_{\beta}) \cdot \sigma \cdot \sqrt{\frac{2}{n}}}$$

This gives the **absolute MDE**. For relative MDE, divide by the baseline mean $\mu$:

$$\text{MDE}_{\text{relative}} = \frac{\text{MDE}}{\mu} = (z_{\alpha/2} + z_{\beta}) \cdot \frac{\sigma}{\mu} \cdot \sqrt{\frac{2}{n}}$$

In [ ]:
def calculate_mde(n, sigma, alpha=0.05, beta=0.2, mu=None):
    """
    Calculate the Minimum Detectable Effect given a fixed sample size.
    
    Parameters
    ----------
    n : int
        Sample size per group.
    sigma : float
        Standard deviation of the metric.
    alpha : float
        Significance level.
    beta : float
        Type II error rate (1 - power).
    mu : float, optional
        Baseline mean (if provided, also returns relative MDE).
    
    Returns
    -------
    dict
        Dictionary with absolute MDE (and relative MDE if mu provided).
    """
    z_alpha_half = stats.norm.ppf(1 - alpha / 2)
    z_beta = stats.norm.ppf(1 - beta)
    
    mde_absolute = (z_alpha_half + z_beta) * sigma * np.sqrt(2 / n)
    
    result = {'mde_absolute': mde_absolute}
    if mu is not None:
        result['mde_relative'] = mde_absolute / mu
        result['mde_relative_pct'] = f"{mde_absolute / mu * 100:.2f}%"
    
    return result

In [ ]:
# Example: what can we detect with n=10,000 per group?
mu_rev = np.mean(revenue)
sigma_rev = np.std(revenue, ddof=1)

mde_result = calculate_mde(n=10000, sigma=sigma_rev, mu=mu_rev)
print(f"With n=10,000 per group:")
print(f"  Absolute MDE: ${mde_result['mde_absolute']:.2f}")
print(f"  Relative MDE: {mde_result['mde_relative_pct']}")
print()

# Compare multiple sample sizes
sample_sizes = [5000, 10000, 25000, 50000, 100000, 175000, 250000]
print(f"{'n per group':>12} | {'Absolute MDE':>14} | {'Relative MDE':>14}")
print("-" * 48)
for n in sample_sizes:
    mde = calculate_mde(n=n, sigma=sigma_rev, mu=mu_rev)
    print(f"{n:>12,} | ${mde['mde_absolute']:>12.2f} | {mde['mde_relative_pct']:>14}")

---

## 6. QuickCart Planning Example

### The Scenario

QuickCart has **50,000 daily active users** who make purchases. They're planning to test a new checkout flow. The question: how long should they run the experiment?

Let's generate realistic user-level revenue data and plan the experiment.

In [ ]:
# Generate realistic daily user revenue data for QuickCart
np.random.seed(123)
daily_users = 50000

# User-level revenue: mixture of behaviors
# - 70% make small purchases ($20-$80)
# - 25% make medium purchases ($80-$200)
# - 5% make large purchases ($200-$1000+)

n_small = int(daily_users * 0.70)
n_medium = int(daily_users * 0.25)
n_large = daily_users - n_small - n_medium

rev_small = np.random.lognormal(mean=3.5, sigma=0.5, size=n_small)    # mean ~$40
rev_medium = np.random.lognormal(mean=4.5, sigma=0.4, size=n_medium)  # mean ~$100
rev_large = np.random.lognormal(mean=5.5, sigma=0.6, size=n_large)    # mean ~$300

user_revenue = np.concatenate([rev_small, rev_medium, rev_large])
np.random.shuffle(user_revenue)

df_quickcart = pd.DataFrame({'user_revenue': user_revenue})

print("QuickCart Daily User Revenue Profile")
print("=" * 45)
print(df_quickcart['user_revenue'].describe())
print(f"\nTotal daily revenue: ${user_revenue.sum():,.0f}")
print(f"CV (sigma/mu): {np.std(user_revenue, ddof=1) / np.mean(user_revenue):.3f}")

In [ ]:
# What can QuickCart detect with different experiment durations?
mu_qc = np.mean(user_revenue)
sigma_qc = np.std(user_revenue, ddof=1)

# Users per group = daily_users / 2 * days (50/50 split)
durations_days = [3, 5, 7, 10, 14, 21, 28]

print(f"QuickCart Experiment Planning")
print(f"Daily users: {daily_users:,} | Users per group per day: {daily_users // 2:,}")
print(f"Baseline mean: ${mu_qc:.2f} | Std: ${sigma_qc:.2f} | CV: {sigma_qc/mu_qc:.3f}")
print()
print(f"{'Duration':>10} | {'n per group':>12} | {'Absolute MDE':>14} | {'Relative MDE':>14}")
print("-" * 60)

mde_values = []
for days in durations_days:
    n_per_group = (daily_users // 2) * days
    mde = calculate_mde(n=n_per_group, sigma=sigma_qc, mu=mu_qc)
    mde_values.append(mde['mde_relative'] * 100)
    print(f"{days:>7} days | {n_per_group:>12,} | ${mde['mde_absolute']:>12.2f} | {mde['mde_relative_pct']:>14}")

In [ ]:
# Visualize MDE vs experiment duration
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(durations_days, mde_values, 'bo-', linewidth=2, markersize=8)

# Annotate key points
for i, (d, m) in enumerate(zip(durations_days, mde_values)):
    ax.annotate(f'{m:.1f}%', (d, m), textcoords='offset points', 
                xytext=(10, 5), fontsize=10)

# Mark typical thresholds
ax.axhline(5, color='green', linestyle='--', alpha=0.5, label='5% MDE (easy to detect)')
ax.axhline(2, color='orange', linestyle='--', alpha=0.5, label='2% MDE (typical goal)')
ax.axhline(1, color='red', linestyle='--', alpha=0.5, label='1% MDE (ambitious)')

# Mark 1-week and 2-week
ax.axvline(7, color='gray', linestyle=':', alpha=0.5)
ax.axvline(14, color='gray', linestyle=':', alpha=0.5)
ax.text(7, ax.get_ylim()[1] * 0.95, '1 week', ha='center', fontsize=9, color='gray')
ax.text(14, ax.get_ylim()[1] * 0.95, '2 weeks', ha='center', fontsize=9, color='gray')

ax.set_xlabel('Experiment Duration (days)')
ax.set_ylabel('Minimum Detectable Effect (%)')
ax.set_title('QuickCart: MDE vs Experiment Duration\n(50K daily users, 50/50 split, 80% power, alpha=0.05)')
ax.legend(loc='upper right')
ax.grid(alpha=0.3)
ax.set_ylim(0, max(mde_values) * 1.1)

plt.tight_layout()
plt.show()

### Planning Conclusion

For QuickCart's revenue metric:
- **1-week experiment**: Can detect ~3-4% relative lift
- **2-week experiment**: Can detect ~2-3% relative lift
- **4-week experiment**: Can detect ~1.5-2% relative lift

If the product team believes their change will produce at least a 3% lift, a 1-week experiment is sufficient. If they want to detect smaller effects (1-2%), they need either more time OR variance reduction techniques.

---

## 7. Variance Reduction Preview

### The Core Insight

From the sample size formula:

$$n \propto \sigma^2$$

If you can **halve the variance** of your metric, you **halve the required sample size** (or equivalently, reduce MDE by a factor of $\sqrt{2} \approx 0.71$).

This is the motivation behind CUPED, stratification, and other variance reduction techniques we'll cover in Weeks 4-5. For now, let's demonstrate the simplest approach: **outlier filtering (winsorization)**.

### Why Outliers Inflate Variance

Revenue data has heavy tails. A single $5,000 order among 10,000 typical $75 orders contributes disproportionately to variance:

$$\text{Var} = \frac{1}{n-1}\sum(x_i - \bar{x})^2$$

That one outlier's squared deviation $(5000 - 75)^2 \approx 24M$ can dominate the entire sum. Capping it at a reasonable percentile preserves most of the signal while dramatically reducing noise.

In [ ]:
def winsorize(data, lower_pct=0, upper_pct=95):
    """
    Winsorize data by capping at given percentiles.
    Values below lower_pct percentile are set to that percentile value.
    Values above upper_pct percentile are set to that percentile value.
    """
    lower_bound = np.percentile(data, lower_pct)
    upper_bound = np.percentile(data, upper_pct)
    return np.clip(data, lower_bound, upper_bound)


# Compare raw vs winsorized at different percentiles
percentiles = [90, 95, 97, 99, 99.5, 100]

print(f"{'Winsorize at':>14} | {'Mean':>10} | {'Std':>10} | {'Variance':>12} | {'Var Ratio':>10} | {'n for 3% MDE':>14}")
print("-" * 85)

raw_var = np.var(user_revenue, ddof=1)
raw_mu = np.mean(user_revenue)

z_a = stats.norm.ppf(0.975)
z_b = stats.norm.ppf(0.8)

for pct in percentiles:
    if pct == 100:
        winsorized = user_revenue
        label = "No cap (raw)"
    else:
        winsorized = winsorize(user_revenue, upper_pct=pct)
        label = f"{pct}th pctile"
    
    w_mu = np.mean(winsorized)
    w_std = np.std(winsorized, ddof=1)
    w_var = np.var(winsorized, ddof=1)
    var_ratio = w_var / raw_var
    
    # Required n per group for 3% relative effect
    delta = w_mu * 0.03
    n_required = int(np.ceil((z_a + z_b)**2 * 2 * w_var / delta**2))
    
    print(f"{label:>14} | ${w_mu:>8.2f} | ${w_std:>8.2f} | {w_var:>12,.0f} | {var_ratio:>9.3f} | {n_required:>14,}")

In [ ]:
# Visualize the effect of winsorization on the distribution and required sample size
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top left: raw distribution vs winsorized
axes[0, 0].hist(user_revenue, bins=100, alpha=0.5, label='Raw', color='blue', density=True)
win_95 = winsorize(user_revenue, upper_pct=95)
axes[0, 0].hist(win_95, bins=100, alpha=0.5, label='Winsorized (95th)', color='green', density=True)
axes[0, 0].set_xlabel('Revenue ($)')
axes[0, 0].set_ylabel('Density')
axes[0, 0].set_title('Distribution: Raw vs Winsorized')
axes[0, 0].legend()
axes[0, 0].set_xlim(0, 500)

# Top right: variance reduction factor
pcts = np.arange(85, 100.5, 0.5)
var_ratios = []
for p in pcts:
    w = winsorize(user_revenue, upper_pct=p)
    var_ratios.append(np.var(w, ddof=1) / raw_var)

axes[0, 1].plot(pcts, var_ratios, 'b-', linewidth=2)
axes[0, 1].axhline(0.5, color='red', linestyle='--', alpha=0.5, label='50% variance reduction')
axes[0, 1].set_xlabel('Winsorization Percentile')
axes[0, 1].set_ylabel('Variance Ratio (winsorized / raw)')
axes[0, 1].set_title('Variance Reduction vs Cap Percentile')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Bottom left: required sample size at different cap levels
n_required_list = []
for p in pcts:
    w = winsorize(user_revenue, upper_pct=p)
    w_mu = np.mean(w)
    w_var = np.var(w, ddof=1)
    delta = w_mu * 0.03
    n_req = (z_a + z_b)**2 * 2 * w_var / delta**2
    n_required_list.append(n_req)

axes[1, 0].plot(pcts, n_required_list, 'r-', linewidth=2)
axes[1, 0].axhline(175000, color='blue', linestyle='--', alpha=0.5, label='1 week (175K per group)')
axes[1, 0].set_xlabel('Winsorization Percentile')
axes[1, 0].set_ylabel('Required n per group')
axes[1, 0].set_title('Required Sample Size (3% MDE) vs Cap Percentile')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Bottom right: MDE at 1-week duration with different caps
n_one_week = 25000 * 7  # per group for 1 week
mde_list = []
for p in pcts:
    w = winsorize(user_revenue, upper_pct=p)
    w_mu = np.mean(w)
    w_sigma = np.std(w, ddof=1)
    mde_abs = (z_a + z_b) * w_sigma * np.sqrt(2 / n_one_week)
    mde_list.append(mde_abs / w_mu * 100)

axes[1, 1].plot(pcts, mde_list, 'g-', linewidth=2)
axes[1, 1].axhline(2, color='red', linestyle='--', alpha=0.5, label='2% MDE target')
axes[1, 1].set_xlabel('Winsorization Percentile')
axes[1, 1].set_ylabel('MDE (%) for 1-week experiment')
axes[1, 1].set_title('MDE Improvement from Winsorization (1 week)')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate: halving variance halves the required sample size
print("=" * 60)
print("VARIANCE REDUCTION: THE FUNDAMENTAL RELATIONSHIP")
print("=" * 60)
print()

# Original variance
sigma_original = np.std(user_revenue, ddof=1)
var_original = sigma_original**2
mu_original = np.mean(user_revenue)
delta_3pct = mu_original * 0.03

n_original = (z_a + z_b)**2 * 2 * var_original / delta_3pct**2

# Halved variance
var_halved = var_original / 2
n_halved = (z_a + z_b)**2 * 2 * var_halved / delta_3pct**2

print(f"Original variance:  {var_original:,.0f}")
print(f"Required n (3% MDE): {n_original:,.0f} per group")
print()
print(f"Halved variance:    {var_halved:,.0f}")
print(f"Required n (3% MDE): {n_halved:,.0f} per group")
print()
print(f"Ratio: {n_halved / n_original:.3f} (confirming: halving variance halves n)")
print()
print(f"Alternatively, with fixed n={n_one_week:,} per group (1 week):")
mde_orig = (z_a + z_b) * sigma_original * np.sqrt(2 / n_one_week)
mde_reduced = (z_a + z_b) * np.sqrt(var_halved) * np.sqrt(2 / n_one_week)
print(f"  Original MDE:     {mde_orig/mu_original*100:.2f}%")
print(f"  Reduced MDE:      {mde_reduced/mu_original*100:.2f}%")
print(f"  MDE ratio:        {mde_reduced/mde_orig:.3f} (= 1/sqrt(2) = {1/np.sqrt(2):.3f})")

### Winsorization: A Simple Starting Point

Winsorization (capping values at a percentile) is the crudest form of variance reduction. Its advantages:
- Simple to implement and explain
- Reduces the influence of outliers
- Can significantly reduce variance for heavy-tailed metrics

Its limitations:
- You lose information about the true tails
- The cap percentile is somewhat arbitrary
- It biases the point estimate (capped mean != true mean)
- More sophisticated methods (CUPED, stratification) achieve better variance reduction without these drawbacks

:::{admonition} When to Use Winsorization
:class: dropdown

**Use winsorization when:**
- Your metric has obvious outliers (data entry errors, bot traffic, B2B customers in a B2C metric)
- You need a quick variance reduction without infrastructure investment
- The outliers are not the effect you're trying to measure

**Avoid winsorization when:**
- Your treatment might specifically affect the tails (e.g., high-value user targeting)
- The "outliers" are legitimate signal (e.g., a feature that helps power users spend more)
- You have access to pre-experiment covariates (use CUPED instead)

**Best practices:**
- Choose the cap percentile BEFORE seeing the experiment results (pre-register it)
- 95th or 99th percentile are common choices
- Apply the SAME cap to both control and treatment
- Report both winsorized and raw results for transparency
:::

In [ ]:
# Simulate a full experiment with and without winsorization
# to confirm that winsorization improves detection power

np.random.seed(42)
n_simulations = 2000
true_effect = 0.03  # 3% lift
n_per_group = 25000  # less than what's needed for raw metric

detected_raw = 0
detected_win95 = 0
detected_win99 = 0

for _ in range(n_simulations):
    # Generate control (same distribution as QuickCart data)
    n_s = int(n_per_group * 0.70)
    n_m = int(n_per_group * 0.25)
    n_l = n_per_group - n_s - n_m
    
    control = np.concatenate([
        np.random.lognormal(3.5, 0.5, n_s),
        np.random.lognormal(4.5, 0.4, n_m),
        np.random.lognormal(5.5, 0.6, n_l)
    ])
    
    # Treatment: 3% multiplicative lift
    treatment = np.concatenate([
        np.random.lognormal(3.5, 0.5, n_s),
        np.random.lognormal(4.5, 0.4, n_m),
        np.random.lognormal(5.5, 0.6, n_l)
    ]) * (1 + true_effect)
    
    # Test on raw data
    _, p_raw = stats.ttest_ind(treatment, control)
    if p_raw < 0.05:
        detected_raw += 1
    
    # Test on winsorized (95th percentile) data
    combined = np.concatenate([control, treatment])
    cap_95 = np.percentile(combined, 95)
    cap_99 = np.percentile(combined, 99)
    
    _, p_win95 = stats.ttest_ind(
        np.clip(treatment, 0, cap_95), 
        np.clip(control, 0, cap_95)
    )
    if p_win95 < 0.05:
        detected_win95 += 1
    
    # Test on winsorized (99th percentile) data
    _, p_win99 = stats.ttest_ind(
        np.clip(treatment, 0, cap_99), 
        np.clip(control, 0, cap_99)
    )
    if p_win99 < 0.05:
        detected_win99 += 1

print(f"Power comparison (n={n_per_group:,} per group, true effect={true_effect*100}%)")
print(f"=" * 55)
print(f"  Raw data:              {detected_raw/n_simulations:.1%} power")
print(f"  Winsorized (99th pct): {detected_win99/n_simulations:.1%} power")
print(f"  Winsorized (95th pct): {detected_win95/n_simulations:.1%} power")
print()
print(f"Winsorization at 95th percentile gives ~{(detected_win95 - detected_raw)/n_simulations*100:.0f} percentage points more power!")

---

## 8. Practical Guidelines

### Rules of Thumb

1. **Always compute sample size BEFORE starting an experiment.** Running until you see significance ("peeking") inflates false positives.

2. **Use your own historical data to estimate variance.** Don't use textbook values — every product has its own noise level.

3. **Be honest about what effect size matters.** Don't plan for 10% lift if your team would ship at 2%. You'll waste time running underpowered tests.

4. **Account for the "winner's curse."** Detected effects are biased upward (especially when power is low). Plan for effects slightly smaller than what you hope for.

5. **Round up.** Always round your sample size UP, and add a buffer (10-20%) for:
   - Users who drop out or get filtered
   - Weekend/weekday effects (run full weeks)
   - The variance estimate being slightly off

### Common Pitfalls

| Pitfall | Consequence | Solution |
|---------|-------------|----------|
| Using population variance instead of per-user variance | Underestimate required n | Always use individual-level variance |
| Ignoring metric skewness | Variance estimate unstable | Use winsorization or log-transform |
| Planning for one-sided test but running two-sided | Underpowered by ~20% | Be consistent; default to two-sided |
| Forgetting the 2x factor | Need n per GROUP, total is 2n | Double-check: report both per-group and total |
| Not accounting for multiple metrics | Bonferroni correction increases required n | Choose ONE primary metric for power calc |

In [ ]:
# A complete planning function that puts it all together
def plan_experiment(df, metric_name, daily_users, target_effects, 
                    alpha=0.05, beta=0.2, winsorize_pct=None, buffer=0.1):
    """
    Complete experiment planning tool.
    
    Parameters
    ----------
    df : pd.DataFrame
        Historical data.
    metric_name : str
        Column name for the metric.
    daily_users : int
        Number of users entering the experiment per day.
    target_effects : list of float
        Relative effect sizes to plan for.
    alpha, beta : float
        Error rates.
    winsorize_pct : float or None
        If provided, winsorize at this percentile before computing variance.
    buffer : float
        Safety buffer (e.g., 0.1 = 10% extra sample).
    
    Returns
    -------
    pd.DataFrame
        Planning table with sample sizes and durations.
    """
    data = df[metric_name].values.copy()
    
    if winsorize_pct is not None:
        cap = np.percentile(data, winsorize_pct)
        data = np.clip(data, 0, cap)
    
    mu = np.mean(data)
    sigma_sq = np.var(data, ddof=1)
    
    z_a = stats.norm.ppf(1 - alpha / 2)
    z_b = stats.norm.ppf(1 - beta)
    
    users_per_group_per_day = daily_users / 2
    
    results = []
    for effect in target_effects:
        delta = mu * effect
        n_per_group = (z_a + z_b)**2 * 2 * sigma_sq / delta**2
        n_with_buffer = n_per_group * (1 + buffer)
        days_needed = np.ceil(n_with_buffer / users_per_group_per_day)
        # Round to full weeks
        weeks_needed = np.ceil(days_needed / 7)
        
        results.append({
            'effect': f"{effect*100:.1f}%",
            'n_per_group': int(np.ceil(n_with_buffer)),
            'total_n': int(np.ceil(2 * n_with_buffer)),
            'days': int(days_needed),
            'weeks_rounded': int(weeks_needed)
        })
    
    return pd.DataFrame(results)


# Plan QuickCart's experiment
print("QUICKCART EXPERIMENT PLAN")
print("=" * 50)
print("\n--- Without Winsorization ---")
plan_raw = plan_experiment(
    df_quickcart, 'user_revenue', 
    daily_users=50000, 
    target_effects=[0.01, 0.02, 0.03, 0.05]
)
print(plan_raw.to_string(index=False))

print("\n--- With Winsorization at 95th Percentile ---")
plan_win = plan_experiment(
    df_quickcart, 'user_revenue', 
    daily_users=50000, 
    target_effects=[0.01, 0.02, 0.03, 0.05],
    winsorize_pct=95
)
print(plan_win.to_string(index=False))

---

## Summary

| Concept | Key Formula / Insight |
|---------|----------------------|
| **Sample size formula** | $n = (z_{\alpha/2} + z_{\beta})^2 \cdot 2\sigma^2 / \delta^2$ per group |
| **MDE formula** | $\text{MDE} = (z_{\alpha/2} + z_{\beta}) \cdot \sigma \cdot \sqrt{2/n}$ |
| **Effect size dominates** | $n \propto 1/\delta^2$: halving the target effect quadruples required n |
| **Variance is linear** | $n \propto \sigma^2$: halving variance halves required n |
| **MDE scales with $\sqrt{n}$** | Doubling experiment length reduces MDE by only ~30% ($1/\sqrt{2}$) |
| **CV matters** | High coefficient of variation = expensive experiments |
| **Winsorization** | Simple variance reduction: cap at 95th/99th percentile |
| **Always plan ahead** | Compute sample size BEFORE the experiment starts |

### What's Next

In **Week 4**, we'll dive into **CUPED (Controlled-experiment Using Pre-Experiment Data)** — a much more powerful variance reduction technique that uses pre-experiment covariates to reduce noise without introducing bias. Unlike winsorization, CUPED can achieve 20-50% variance reduction while preserving unbiased estimation.